# Preprocessing .mib data for template matching

## Contents
 1. [Load raw data](#1.-Load-raw-data)
 2. [Reshape and set metadata](#2.-Reshape-and-metadata)
 3. [Save converted data](#3.-Save-as-.hspy)
 4. [Load converted data](#4.-Load-.hspy)
 5. [VBF and cropping](#6.-Crop)
 6. [Center data](#7.-Center) 
 7. [Calibrate diffraction space](#8.-Calibrate)
 9. [Save cropped and centered data](#9.-Save-cropped-and-centered-data)
 10. [Load cropped and centered data](#10.-Load-cropped-and-centered-data)
 11. [Background subtraction](#11.-Background-subtraction)

In [ ]:
%matplotlib inline
import hyperspy.api as hs #General hyperspy package
import pyxem as pxm #Electron diffraction tools based on hyperspy
import numpy as np #General numerical and matrix support
import matplotlib.pyplot as plt #Plotting tools
import matplotlib.colors as mcolors #Some plotting color tools
from scipy.ndimage import gaussian_filter #For smoothing the xmap
from skimage import filters, morphology #For DoG filtering and thresholding

from pathlib import Path
import json

#Ignore warnings for cleaner output
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

#Function to read .hdr files and return a dictionary of the header information. Made by Emil F. Christiansen (emil.christiansen@ntnu.no)
def read_hdr(filename):
   
    filename=Path(filename)
    if not filename.suffix == '.hdr':
        raise ValueError(f'Cannot read hdr file: File "{str(filename)}" is not a .hdr file')
    hdr = {}
    with filename.open('r') as f:
        for line in f.readlines():
            content = line.split(':', maxsplit=1)
            if len(content)>1:
                hdr[content[0].strip()] = content[1].strip()
    return hdr

class MyPath(Path): #helpful for appending suffixes to filenames
    _flavour = type(Path())._flavour


    def append(self, s, suffix=None, delimiter='_'):
       
        if suffix:
            return self.with_name(f'{self.stem}{delimiter}{s}.{suffix}')
        else:
            return self.with_stem(f'{self.stem}{delimiter}{s}')

# 1. Load raw data

In [ ]:
# Create path to the .mib file
datapath = MyPath("20260417_Ingrid/20260417_105201/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m7p2_Ty3p8.mib")

In [ ]:
# Three files are in the same folder, .mib with the data, .hdr with the header information and .json with the parameters
mib_data = hs.load(datapath, lazy=True)
hdr = read_hdr(datapath.with_suffix('.hdr'))
parameters = json.load((datapath.parent/'json.json').open('r'))

print(f'MIB :\n{mib_data}\n')

# 2. Reshape and metadata

In [ ]:
# Reshape
scan_shape = (256, 256)
chunksize = 32

data = mib_data.data
data = data.reshape(scan_shape + mib_data.axes_manager.signal_shape)
data = data.rechunk(32, 32, 32, 32)
print(data)

In [ ]:
# Create a PyXem signal from the data, in order to use the tools in PyXem for processing and analysis
signal = pxm.signals.LazyElectronDiffraction2D(data)

In [ ]:
# set metadata from .json file
signal.metadata.General.title='GaAs' #Set the title of the data

#Set the most important experimental parameters
signal.set_experimental_parameters(beam_energy=parameters['Beam energy'],
                                   camera_length=parameters['Cameralength'],
                                   scan_rotation=parameters['Rotation'],
                                   rocking_angle=parameters['PED angle'],
                                   rocking_frequency=parameters['PED frequency'],
                                   exposure_time=parameters['Exposure time']
                                   )

signal.set_scan_calibration(parameters['dx'])

In [ ]:
# Print the metadata to check that it is correct
print(signal.metadata)

In [ ]:
print(signal.axes_manager)

# 3. Save as .hspy

In [ ]:
signal.save(datapath.with_suffix('.hspy'), overwrite=True, chunks=(chunksize,) * 4)
signal = hs.load(str(datapath), lazy=True) #Use the lazy keyword to work with big datasets without loading everything into memory at once

# 4. Load .hspy

In [ ]:
signal = hs.load("20260417_Ingrid/20260417_105201/SPED_CL8_256x256px_w10_3p40nm_ws_10ms_tx_m7p2_Ty3p8.hspy", lazy=True)

In [ ]:
signal = hs.load(datapath.with_suffix('.hspy'), lazy=True)

# 5. Crop
Use vbf to crop

In [ ]:
nx, ny = signal.axes_manager.signal_shape #Get the signal shape
vbf = signal.get_integrated_intensity(hs.roi.CircleROI(nx//2, ny//2, 10))
vbf.compute(norm='symlog')


In [ ]:
vbf.plot(norm='symlog')
left, right, top, bottom = vbf.axes_manager.signal_extent #Set the roi to cover the whole scan area
left, right, top, bottom = 45.370000000000005, 684.0400000000001, 362.96000000000004, 722.4300000000001 #Select a pre-defined region
roi = hs.roi.RectangularROI(left=left, right=right, top=top, bottom=bottom)
roi.add_widget(vbf)

In [ ]:
assert False

In [ ]:
print(roi.left, roi.right, roi.top, roi.bottom)

In [ ]:
# set crop
signal = roi(signal) 

In [ ]:
# verify
signal.plot(norm='symlog')

In [ ]:
signal.save(MyPath(datapath).append('cropped', 'hspy'))

In [ ]:
signal

load cropped hspy signal

In [ ]:
signal = hs.load(datapath.append('cropped', 'hspy'), lazy=True)

# Center

In [ ]:
# make a navigsation mask to exclude the edges of the scan from the centering, as they often contain artifacts and can skew the centering
def make_nav_mask(signal, width=None):
    mask = hs.signals.Signal2D(np.zeros(signal.axes_manager.navigation_shape, dtype=bool).T).T
    if width is not None:
        mask.inav[width:-width, width:-width] = True
    return mask


def center_direct_beam(signal, com_mask=None, estimate_linear_shift=False, plot_results=False, estimate_linear_shifts_kwargs = dict(), **kwargs):
    """
    Center the direct beam of a signal

    :param signal: The signal to center the direct beam for
    :param com_mask: The region of the diffraction patterns to calculate COM within
    :param estimate_linear_shift: Whether to estimate linear shifts or not
    :param plot_results: Whether to plot the results or not
    :param estimate_linear_shifts_kwargs: Keyword arguments passed to linear shift estimation
    :kwargs: Optional keyword arguments passed to signal.center_direct_beam.
    :returns: Centered. The centered signal.

    :type signal: Signal
    :type com_mask: Union[None, tuple]
    :type estimate_linear_shift: bool
    :type plot_results: bool
    :type estimate_linear_shift_kwargs: Dict
    :rtype: Signal
    """

    kwargs['inplace'] = kwargs.get('inplace', False) #Set default inplace to False
    _ = kwargs.pop('method', None) #Remove any `method` as it is not compatible with the shifts we will specify later on.

    # Get the max projection before centering as a reference to evaluate the centering.
    max_before = signal.max(axis=[0, 1]) #Get the max projection before centering
    max_before.metadata.General.title = 'Before' #Set the title of the max projection before centering
    try:
        max_before.compute()
    except Exception:
        pass

    centering_metadata = {} #Metadata dict for centering parameters

    #Set up the parameters used for centering. This is a slightly cumbersome way to do it, but it ensures that all parameters are stored in the metadata.
    if 'shifts' not in kwargs:
        if com_mask is None:
            nx, ny = signal.axes_manager.signal_size #Get the signal shape
            cx, cy = nx//2, ny//2 #Center coordinates
            com_mask = (cx, cy, 15) #15 pixels radius circular mask around the center of the diffraction pattern
        centering_metadata['COM_mask'] = com_mask #Store the COM mask in the metadata

        shifts = signal.get_direct_beam_position(method='center_of_mass', mask=com_mask) #Get the shifts using center of mass
        #Compute the shifts if lazy, as they need to be in memory for later processing.
        try:
            shifts.compute()
        except Exception:
            pass
        kwargs['shifts'] = shifts

    if estimate_linear_shift: #Estimate linear shifts if requested
        linear_shift = kwargs['shifts'].get_linear_plane(**estimate_linear_shifts_kwargs)
        kwargs['shifts'] = linear_shift
        centering_metadata['estimate_linear_shift'] = estimate_linear_shifts_kwargs

    centering_metadata['Shifts'] = kwargs['shifts']
    centered = signal.center_direct_beam(**kwargs) #Center the direct beam

    # Get the max projection after centering to compare with the max projection before the centering to evaluate the centering.
    max_after = centered.max(axis=[0, 1]) #Get the max projection after centering
    max_after.metadata.General.title = 'After' #Set the title of the max projection after centering

    try:
        max_after.compute()
    except Exception:
        pass
        
    centering_metadata['Max_before'] = max_before #Store the max projections before and after centering in the metadata
    centering_metadata['Max_after'] = max_after #Store the max projections before and after centering in the metadata

    #Add centering metadata to signal metadata.
    centered.metadata.add_dictionary({
        'Preprocessing': {
            'Centering': centering_metadata
        }
    })

    if plot_results: #Plot a comparison of the maximum projection before and after the centering if requested.
        hs.plot.plot_images([max_before, max_after], overlay=True, alphas=[1, 0.75], colors=['w', 'r'])

    return centered
    

In [ ]:
dp_max = signal.max(axis=[0, 1])
#dp_max.plot(norm='symlog')
x0, y0 = dp_max.axes_manager.signal_shape
x0, y0 = x0//2, y0//2
roi = hs.roi.CircleROI(x0, y0, 10)
#roi.add_widget(dp_max)

In [ ]:
shifted_signal = center_direct_beam(signal, (roi.cx, roi.cy, roi.r), estimate_linear_shift=True, plot_results=True)

In [ ]:
nx, ny = shifted_signal.axes_manager.signal_shape
nx, ny = nx/2.0, ny/2.0
shifted_signal_2 = center_direct_beam(shifted_signal, (nx, ny, roi.r,), estimate_linear_shift=True, plot_results=True)

In [ ]:
signal = shifted_signal_2

# Calibrate

In [ ]:
signal.set_diffraction_calibration(1.0)
scale = 0.013178278600334982981

In [ ]:
signal.set_diffraction_calibration(scale)

# Save crop and centered data

In [ ]:
signal.save(datapath.append('big_centered', 'hspy'))

# Load crop and centered data

In [ ]:
signal = hs.load(datapath.append('big_centered', 'hspy'), lazy=True)

In [ ]:
assert False

In [ ]:
signal_mean = signal.mean(axis=[0, 1])
signal_mean.plot(norm='symlog', cmap='inferno', colorbar=None, axes_off=True)

# Filtering

In [ ]:
mask = signal.get_direct_beam_mask(radius=7)

signal_masked = signal * ~mask

dp_dog = signal_masked.subtract_diffraction_background(
    "difference of gaussians",
    inplace=False,
    min_sigma=0.8,
    max_sigma=3.2,
)

In [ ]:
dp_mean = signal_masked.mean(axis=[0, 1])
dp_mean.plot(norm='symlog', cmap='inferno', colorbar=None, axes_off=True)

In [ ]:
signal.plot(norm='symlog', colorbar=None, axes_off=True)

In [ ]:
signal = signal_masked

signal.save(datapath.append('big_centered_masked', 'hspy'))